# Purpose

The main aim of the notebook is to do basic cleaning and to prepare a content filtering mechanism.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
root_path = Path().cwd().parent
songs_data_path = root_path / "data" / "raw" / "songs-info.csv"

In [3]:
df_songs = pd.read_csv(songs_data_path)
df_songs.head()

,track_id,name,artist,pulse_play_preview_url,pulse_play_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,TRIOREW128F424EAF0,Mr. Brightside,The Killers,https://p.scdn.co/mp3-preview/4d26180e6961fd46...,09ZQ5TmUG8TSL56n0knqrj,"rock, alternative, indie, alternative_rock, in...",NaN,2004,222200,0.355,...,1,-4.360,1,0.0746,0.001190,0.000000,0.0971,0.240,148.114,4
1,TRRIVDJ128F429B0E8,Wonderwall,Oasis,https://p.scdn.co/mp3-preview/d012e536916c927b...,06UfBBDISthj1ZJAtX4xjj,"rock, alternative, indie, pop, alternative_roc...",NaN,2006,258613,0.409,...,2,-4.373,1,0.0336,0.000807,0.000000,0.2070,0.651,174.426,4
2,TROUVHL128F426C441,Come as You Are,Nirvana,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,0keNu0t0tqsWtExGM3nT1D,"rock, alternative, alternative_rock, 90s, grunge",RnB,1991,218920,0.508,...,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
3,TRUEIND128F93038C4,Take Me Out,Franz Ferdinand,https://p.scdn.co/mp3-preview/399c401370438be4...,0ancVQ9wEcHVd0RrGICTE4,"rock, alternative, indie, alternative_rock, in...",NaN,2004,237026,0.279,...,9,-8.851,1,0.0371,0.000389,0.000655,0.1330,0.490,104.560,4
4,TRLNZBD128F935E4D8,Creep,Radiohead,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,01QoK9DA7VTeTSE3MNzp4I,"rock, alternative, indie, alternative_rock, in...",RnB,2008,238640,0.515,...,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4


In [4]:
df_songs.info()

<class 'pandas.DataFrame'>
RangeIndex: 50683 entries, 0 to 50682
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   track_id                50683 non-null  str    
 1   name                    50683 non-null  str    
 2   artist                  50683 non-null  str    
 3   pulse_play_preview_url  50683 non-null  str    
 4   pulse_play_id           50683 non-null  str    
 5   tags                    49556 non-null  str    
 6   genre                   22348 non-null  str    
 7   year                    50683 non-null  int64  
 8   duration_ms             50683 non-null  int64  
 9   danceability            50683 non-null  float64
 10  energy                  50683 non-null  float64
 11  key                     50683 non-null  int64  
 12  loudness                50683 non-null  float64
 13  mode                    50683 non-null  int64  
 14  speechiness             50683 non-null  float64
 

In [5]:
df_songs.shape

(50683, 21)

In [6]:
# missing values

df_songs.isna().sum()

track_id                      0
name                          0
artist                        0
pulse_play_preview_url        0
pulse_play_id                 0
tags                       1127
genre                     28335
year                          0
duration_ms                   0
danceability                  0
energy                        0
key                           0
loudness                      0
mode                          0
speechiness                   0
acousticness                  0
instrumentalness              0
liveness                      0
valence                       0
tempo                         0
time_signature                0
dtype: int64

In [7]:
df_songs.isna().mean().mul(100).sort_values(ascending=False).head(2)

genre    55.906320
tags      2.223625
dtype: float64

In [8]:
df_songs.duplicated(subset=["pulse_play_id","year","duration_ms"]).sum()

np.int64(9)

__Dropping Duplicates.__

In [9]:
# drop duplicates

df_songs.drop_duplicates(subset=["pulse_play_id","year","duration_ms"],inplace=True)

In [10]:
df_songs.duplicated(subset=["pulse_play_id","year","duration_ms"]).sum()

np.int64(0)

__Removing Columns that are not Required for Collabortive Filtering.__

In [11]:
cols_to_remove = ["track_id","name","pulse_play_preview_url","pulse_play_id","genre"]


df_colab_filtering = df_songs.drop(columns=cols_to_remove)

df_colab_filtering

,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,The Killers,"rock, alternative, indie, alternative_rock, in...",2004,222200,0.355,0.918,1,-4.360,1,0.0746,0.001190,0.000000,0.0971,0.240,148.114,4
1,Oasis,"rock, alternative, indie, pop, alternative_roc...",2006,258613,0.409,0.892,2,-4.373,1,0.0336,0.000807,0.000000,0.2070,0.651,174.426,4
2,Nirvana,"rock, alternative, alternative_rock, 90s, grunge",1991,218920,0.508,0.826,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
3,Franz Ferdinand,"rock, alternative, indie, alternative_rock, in...",2004,237026,0.279,0.664,9,-8.851,1,0.0371,0.000389,0.000655,0.1330,0.490,104.560,4
4,Radiohead,"rock, alternative, indie, alternative_rock, in...",2008,238640,0.515,0.430,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50678,アンティック-珈琲店-,NaN,2008,273440,0.438,0.933,6,-3.062,0,0.1650,0.003120,0.000000,0.1300,0.421,166.956,4
50679,ACIDMAN,"rock, alternative_rock, japanese, cover",2004,275133,0.351,0.693,0,-6.811,1,0.1200,0.000940,0.000049,0.1920,0.450,200.350,4
50680,coldrain,"metal, metalcore, post_hardcore",2014,254826,0.434,0.975,10,-3.092,0,0.2680,0.000108,0.001410,0.1630,0.282,158.025,4
50681,アンティック-珈琲店-,NaN,2008,243293,0.513,0.902,4,-3.914,0,0.0530,0.000715,0.001350,0.0571,0.618,109.923,4


In [12]:
# check for missing values

df_colab_filtering.isna().sum()

artist                 0
tags                1126
year                   0
duration_ms            0
danceability           0
energy                 0
key                    0
loudness               0
mode                   0
speechiness            0
acousticness           0
instrumentalness       0
liveness               0
valence                0
tempo                  0
time_signature         0
dtype: int64

In [13]:
# fill the tags column missing values with string "no_tags"

df_colab_filtering.fillna({"tags":"no_tags"},inplace=True)

,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,The Killers,"rock, alternative, indie, alternative_rock, in...",2004,222200,0.355,0.918,1,-4.360,1,0.0746,0.001190,0.000000,0.0971,0.240,148.114,4
1,Oasis,"rock, alternative, indie, pop, alternative_roc...",2006,258613,0.409,0.892,2,-4.373,1,0.0336,0.000807,0.000000,0.2070,0.651,174.426,4
2,Nirvana,"rock, alternative, alternative_rock, 90s, grunge",1991,218920,0.508,0.826,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
3,Franz Ferdinand,"rock, alternative, indie, alternative_rock, in...",2004,237026,0.279,0.664,9,-8.851,1,0.0371,0.000389,0.000655,0.1330,0.490,104.560,4
4,Radiohead,"rock, alternative, indie, alternative_rock, in...",2008,238640,0.515,0.430,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50678,アンティック-珈琲店-,no_tags,2008,273440,0.438,0.933,6,-3.062,0,0.1650,0.003120,0.000000,0.1300,0.421,166.956,4
50679,ACIDMAN,"rock, alternative_rock, japanese, cover",2004,275133,0.351,0.693,0,-6.811,1,0.1200,0.000940,0.000049,0.1920,0.450,200.350,4
50680,coldrain,"metal, metalcore, post_hardcore",2014,254826,0.434,0.975,10,-3.092,0,0.2680,0.000108,0.001410,0.1630,0.282,158.025,4
50681,アンティック-珈琲店-,no_tags,2008,243293,0.513,0.902,4,-3.914,0,0.0530,0.000715,0.001350,0.0571,0.618,109.923,4


In [14]:
# check for missing values

df_colab_filtering.isna().sum()

artist              0
tags                0
year                0
duration_ms         0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
dtype: int64

In [15]:
# artist names as lower case

df_colab_filtering["artist"] = df_colab_filtering["artist"].str.lower()

In [16]:
df_colab_filtering['artist'].nunique()

8317

In [17]:
df_songs.loc[:,'year'].nunique()

75

In [18]:
df_songs.select_dtypes(include='number').agg(['min','max'])

,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
min,1900,1439,0.000,0.0,0,-60.000,0,0.000,0.000,0.000,0.000,0.000,0.000,0
max,2022,3816373,0.986,1.0,11,3.642,1,0.954,0.996,0.999,0.999,0.993,238.895,5


In [19]:
df_songs.loc[:,'tags'].str.lower().str.split(',').explode().str.strip().value_counts()

tags
rock            10681
indie            7284
electronic       6592
alternative      6271
pop              4650
                ...  
dark_ambient      602
japanese          489
polish            411
j_pop             213
russian           127
Name: count, Length: 100, dtype: int64

# Basic Cleaning Completed -> Onto Content Filtering

In [20]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from category_encoders.count import CountEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer

In [21]:
df_colab_filtering.shape

(50674, 16)

In [22]:
df_colab_filtering.info()

<class 'pandas.DataFrame'>
Index: 50674 entries, 0 to 50682
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   artist            50674 non-null  str    
 1   tags              50674 non-null  str    
 2   year              50674 non-null  int64  
 3   duration_ms       50674 non-null  int64  
 4   danceability      50674 non-null  float64
 5   energy            50674 non-null  float64
 6   key               50674 non-null  int64  
 7   loudness          50674 non-null  float64
 8   mode              50674 non-null  int64  
 9   speechiness       50674 non-null  float64
 10  acousticness      50674 non-null  float64
 11  instrumentalness  50674 non-null  float64
 12  liveness          50674 non-null  float64
 13  valence           50674 non-null  float64
 14  tempo             50674 non-null  float64
 15  time_signature    50674 non-null  int64  
dtypes: float64(9), int64(5), str(2)
memory usage: 6.6 MB


__Most Important Step Below.__

In [23]:
# cols to transform

frequency_encode_cols = ['year']
ohe_cols = ['artist',"time_signature","key"]
tfidf_col = 'tags'
standard_scale_cols = ["duration_ms","loudness","tempo"]
min_max_scale_cols = ["danceability","energy","speechiness","acousticness","instrumentalness","liveness","valence"]

In [24]:
transformer  = ColumnTransformer(
    transformers= [
        ("frequency_encode", CountEncoder(normalize=True,return_df=True),frequency_encode_cols),
        ("ohe_cols" , OneHotEncoder(handle_unknown="ignore") , ohe_cols),
        ('tf-idf',TfidfVectorizer(max_features=85),tfidf_col),
        ('Standard Scaler', StandardScaler(),standard_scale_cols),
        ('MinMaxScaler',MinMaxScaler(),min_max_scale_cols)
    ],remainder='passthrough',n_jobs=-1
)

In [25]:
transformer

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('frequency_encode', ...), ('ohe_cols', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer

In [26]:
# fit the transformer

transformer.fit(df_colab_filtering)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('frequency_encode', ...), ('ohe_cols', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer

In [27]:
# transform the data

transformed_df = transformer.transform(df_colab_filtering)

In [28]:
transformed_df.shape

(50674, 8431)

In [29]:
transformed_df

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 907911 stored elements and shape (50674, 8431)>

# Calculating Similarity

In [30]:
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
# fetch a song where artist is Shakira

df_songs.loc[df_songs["artist"] == "Shakira"]

,track_id,name,artist,pulse_play_preview_url,pulse_play_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,TRLWDVU128F932B093,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...,07PHBDuUmOeZ7jeKSbAbKi,"rock, pop, female_vocalists, singer_songwriter...",NaN,2012,196826,0.787,...,1,-4.967,0,0.0474,0.29800,0.000005,0.2060,0.860,107.674,4
2068,TRILOWN128F426080F,Underneath Your Clothes,Shakira,https://p.scdn.co/mp3-preview/6c5a56058ce04371...,07qRl4PT2lA6O3KN40McLz,"rock, pop, female_vocalists, singer_songwriter...",Pop,2013,224893,0.707,...,8,-5.293,1,0.0298,0.69100,0.000000,0.1030,0.407,82.784,4
2205,TRXLMFJ12903CC06F7,She Wolf,Shakira,https://p.scdn.co/mp3-preview/4dc802fd3b06fcb5...,075xFXR0JDBwFPVinG1ig5,"electronic, pop, female_vocalists, experimenta...",NaN,2009,187866,0.857,...,7,-6.480,1,0.0428,0.32300,0.003220,0.3140,0.868,121.994,4
2469,TRSQAWU128F92EA20F,La Tortura,Shakira,https://p.scdn.co/mp3-preview/e62ae4649f029729...,0ofDrTTcinCUxm7wqCLPQa,"electronic, pop, female_vocalists, singer_song...",Latin,2011,213106,0.741,...,0,-5.904,1,0.0421,0.02300,0.000788,0.1200,0.834,100.001,4
3374,TRINUNP12903CD84D9,Did It Again,Shakira,https://p.scdn.co/mp3-preview/5477eae2283113ff...,0eMNEdcC5OImvrfn79J9dU,"electronic, pop, female_vocalists, experimenta...",NaN,2009,227333,0.869,...,5,-5.069,0,0.0896,0.50900,0.000000,0.0741,0.599,137.955,4
4319,TROLUWR128F92E5858,Te Dejo Madrid,Shakira,https://p.scdn.co/mp3-preview/bcaccba847a5cf53...,0IESLhxv5iqXvMH5mm3z88,"pop, experimental, singer_songwriter, dance, l...",NaN,2001,187333,0.741,...,11,-5.011,1,0.0379,0.02440,0.000015,0.2830,0.926,131.017,4
4600,TRLNLES128F932DA8E,Fool,Shakira,https://p.scdn.co/mp3-preview/4ce283ae87d032f5...,0MttJjfO3pkTHJgVdXqPcP,"rock, pop, female_vocalists, acoustic, pop_rock",Rap,2001,230333,0.640,...,3,-5.848,1,0.0241,0.04560,0.000000,0.0988,0.420,100.994,4
4618,TRUIPZL12903CA0BFE,Rules,Shakira,https://p.scdn.co/mp3-preview/e5c557dc2b36006a...,1f7TZQzs4UikFQIzeWOtOj,"pop, female_vocalists, singer_songwriter, pop_...",NaN,2001,219106,0.692,...,5,-6.365,0,0.0418,0.03590,0.000002,0.1850,0.775,149.000,4
6047,TRAAKDG128F42A0ECB,Hips Don't Lie,Shakira,https://p.scdn.co/mp3-preview/3859547944f57cfb...,01Yj2MCGpjZs34PRlGgz4K,"pop, female_vocalists, singer_songwriter, danc...",Pop,2001,217453,0.777,...,10,-5.867,0,0.0734,0.28400,0.000000,0.4300,0.760,100.003,4
6819,TRBAHID128F4278EAF,Objection (Tango),Shakira,https://p.scdn.co/mp3-preview/bf65095d5ce58358...,0p9QhtUdbyDAQ6k14hQ2i3,"pop, female_vocalists, singer_songwriter, danc...",Pop,2001,222533,0.603,...,11,-5.282,0,0.0677,0.01470,0.000000,0.0246,0.705,179.344,4


__Preparing Songs Input.__

In [32]:
df_songs[df_songs["name"] == "Whenever, Wherever"]

,track_id,name,artist,pulse_play_preview_url,pulse_play_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,TRLWDVU128F932B093,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...,07PHBDuUmOeZ7jeKSbAbKi,"rock, pop, female_vocalists, singer_songwriter...",NaN,2012,196826,0.787,...,1,-4.967,0,0.0474,0.298,0.000005,0.206,0.86,107.674,4


In [33]:
songs_input = df_colab_filtering[df_songs["name"] == "Whenever, Wherever"]

In [34]:
songs_input

,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,shakira,"rock, pop, female_vocalists, singer_songwriter...",2012,196826,0.787,0.828,1,-4.967,0,0.0474,0.298,0.000005,0.206,0.86,107.674,4


In [37]:
input_vector = transformer.transform(songs_input)

In [38]:
input_vector.shape

(1, 8431)

In [39]:
transformed_df.shape

(50674, 8431)

In [41]:
similarity_scores = cosine_similarity(transformed_df,input_vector)

In [45]:
similarity_scores.shape

(50674, 1)

__Remember__: ravel() is used for flattening the array

In [55]:
similarity_scores.ravel()

array([0.99999914, 0.99999847, 0.99999921, ..., 0.99999877, 0.9999992 ,
       0.99999891], shape=(50674,))

In [58]:
top_10_song_indexes = np.argsort(similarity_scores.ravel())[-11:][::-1]

In [59]:
top_10_song_indexes

array([ 1025, 12305,  6046,  6129, 17241,  6133,  7172,  6121,  6526,
       38383,  6287])

In [63]:
top_10_songs = df_songs.iloc[top_10_song_indexes]

In [64]:
top_10_songs

,track_id,name,artist,pulse_play_preview_url,pulse_play_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,TRLWDVU128F932B093,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...,07PHBDuUmOeZ7jeKSbAbKi,"rock, pop, female_vocalists, singer_songwriter...",NaN,2012,196826,0.787,...,1,-4.967,0,0.0474,0.29800,0.000005,0.2060,0.860,107.674,4
12306,TRYFVKK128F4240FE8,Why Wait,Shakira,https://p.scdn.co/mp3-preview/d78c90c5cb5626be...,0HiJFRxWme9myvUiDlqQ8q,"pop, experimental, singer_songwriter, dance",NaN,2001,221240,0.887,...,1,-5.535,0,0.0431,0.14400,0.000590,0.1230,0.399,129.943,4
6047,TRAAKDG128F42A0ECB,Hips Don't Lie,Shakira,https://p.scdn.co/mp3-preview/3859547944f57cfb...,01Yj2MCGpjZs34PRlGgz4K,"pop, female_vocalists, singer_songwriter, danc...",Pop,2001,217453,0.777,...,10,-5.867,0,0.0734,0.28400,0.000000,0.4300,0.760,100.003,4
6130,TRBAUVN128F932FEF8,Oops!...I Did It Again,Britney Spears,https://p.scdn.co/mp3-preview/7fb86827422540ad...,095uakqDYR50Uza0mxvPWB,"pop, female_vocalists, dance, 00s",Pop,2014,211786,0.751,...,1,-5.351,0,0.0435,0.34000,0.000018,0.2550,0.886,95.045,4
17243,TRMBDIR128F4279C1F,Perfect Lover,Britney Spears,https://p.scdn.co/mp3-preview/52671e54d36f077e...,1BhxPx4evrx8X02RHGrLdi,"pop, dance, rnb, 00s",Rock,2007,182680,0.718,...,1,-3.959,0,0.0360,0.35300,0.000390,0.1020,0.805,117.067,4
6134,TRDXCSH128F92ED4A1,Bootylicious,Destiny's Child,https://p.scdn.co/mp3-preview/7e327ccb1e4c52b2...,09mkdGhqb5ySKVsnkx9hy2,"pop, female_vocalists, dance, soul, hip_hop, r...",NaN,2001,207733,0.835,...,1,-4.364,0,0.2840,0.00247,0.000000,0.1710,0.586,103.358,4
7173,TRLQBEN128F92E7415,Wild Things,Alessia Cara,https://p.scdn.co/mp3-preview/c13f00088525d0b2...,0RgHkNpqtAHGGBVro6mmqD,"pop, female_vocalists",Country,2016,188493,0.741,...,1,-5.362,0,0.1080,0.01950,0.000000,0.0822,0.735,107.960,4
6122,TRGZIMZ128F930A016,La Isla Bonita,Madonna,https://p.scdn.co/mp3-preview/d8f3cafe99c1f0cd...,0rpndqrkU9y9nckNCfjcq6,"pop, female_vocalists, dance, 80s",NaN,2009,242946,0.708,...,1,-4.736,0,0.0362,0.39200,0.000001,0.0561,0.968,99.953,4
6527,TRXHWIA128E078A9BB,Cruel Summer,Bananarama,https://p.scdn.co/mp3-preview/47d13ef240a58bef...,0BmUCeyXpTrUVahHKVFxuB,"pop, female_vocalists, dance, 80s, new_wave",NaN,1984,209573,0.665,...,1,-5.828,0,0.0278,0.25000,0.008550,0.0537,0.932,108.429,4
38389,TRWUWRZ128F42ADA4A,Dreams for Plans,Shakira,https://p.scdn.co/mp3-preview/6e2c021846087a88...,2ObxMmMaDINr0ynkqW2BlY,"pop, female_vocalists, guitar, pop_rock",Pop,2005,242760,0.689,...,1,-7.427,1,0.0286,0.18000,0.000038,0.0844,0.548,96.098,4


In [74]:
def recommend(song_name, songs_data, transformed_data, k=10):
    """
    Recommends top k songs similar to the given song based on content-based filtering.

    Parameters:
    song_name (str): The name of the song to base the recommendations on.
    songs_data (DataFrame): The DataFrame containing song information.
    transformed_data (ndarray): The transformed data matrix for similarity calculations.
    k (int, optional): The number of similar songs to recommend. Default is 10.

    Returns:
    DataFrame: A DataFrame containing the top k recommended songs with their names, artists, and Pulse Play preview URLs.
    """
    # filter out the song from data
    song_row = songs_data.loc[songs_data["name"] == song_name,:]
    if song_row.empty:
        print("This Song is currenlty unavailable in the Platform.")
    else:
        # get the index of song
        song_index = song_row.index[0]
        print(song_index)
        # generate the input vector
        input_vector = transformed_data[song_index].reshape(1,-1)
        # calculate similarity scores
        similarity_scores = cosine_similarity(input_vector, transformed_data)
        print(similarity_scores.shape)
        # get the top k songs
        top_k_songs_indexes = np.argsort(similarity_scores.ravel())[-k-1:][::-1]
        print(top_k_songs_indexes)
        # get the top k songs names
        top_k_songs_names = songs_data.iloc[top_k_songs_indexes]
        # print the top k songs
        top_k_list = top_k_songs_names[['name','artist','pulse_play_preview_url']].reset_index(drop=True)
        return top_k_list

In [75]:
# recommend song using the function

recommend("Whenever, Wherever",songs_data=df_songs,transformed_data=transformed_df,k=10)

1025
(1, 50674)
[ 1025 12305  6046  6129 17241  6133  7172  6121  6526 38383  6287]


,name,artist,pulse_play_preview_url
0,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...
1,Why Wait,Shakira,https://p.scdn.co/mp3-preview/d78c90c5cb5626be...
2,Hips Don't Lie,Shakira,https://p.scdn.co/mp3-preview/3859547944f57cfb...
3,Oops!...I Did It Again,Britney Spears,https://p.scdn.co/mp3-preview/7fb86827422540ad...
4,Perfect Lover,Britney Spears,https://p.scdn.co/mp3-preview/52671e54d36f077e...
5,Bootylicious,Destiny's Child,https://p.scdn.co/mp3-preview/7e327ccb1e4c52b2...
6,Wild Things,Alessia Cara,https://p.scdn.co/mp3-preview/c13f00088525d0b2...
7,La Isla Bonita,Madonna,https://p.scdn.co/mp3-preview/d8f3cafe99c1f0cd...
8,Cruel Summer,Bananarama,https://p.scdn.co/mp3-preview/47d13ef240a58bef...
9,Dreams for Plans,Shakira,https://p.scdn.co/mp3-preview/6e2c021846087a88...


# Content-Based Filtering Recommendation Flow

## Step 1: User Provides Input

User enters:

- Song Name
- Artist Name

Example:

```text
Song Name : Waka Waka
Artist Name : Shakira
```

↓

## Step 2: Search Song Metadata Dataset

Search the songs dataset in the __Original and Cleaned Songs Info Dataset__ and find the matching row.

__Note: The Songs Meta data should be cleaned. But the Columns of Song Name, Track ID , Preview URL must be present.__

```text
Song Name + Artist Name
            ↓
      Matching Row
```

Example:

| artist | song_name | year | tags |
|----------|----------|------|------|
| Shakira | Waka Waka | 2012 | rock pop female_vocalists singer_songwriter |

↓

## Step 3: Extract Row Index

Retrieve the row index corresponding to the selected song.

Example:

```python
index = 1025
```

```text
Song + Artist
        ↓
Row Extract
        ↓
Index
```

↓

## Step 4: Retrieve Corresponding Vector from Transformed Matrix

The entire song dataset has already been transformed using:

- Frequency Encoding
- One Hot Encoding
- TF-IDF
- Scaling

Result:

```python
transformed_matrix.shape

(50674, 2000)
```

where

```text
Rows    → Songs
Columns → Features
```

Using the index:

```python
input_vector = transformed_matrix[1025]
```

```text
Index
    ↓
Transformed Matrix
    ↓
Input Vector
```

↓

## Step 5: Compute Similarity Scores

Compare the input vector against all songs.

Using:

```python
cosine_similarity(transformed_matrix,input_vector)
```

Output:

```text
Song A : 1.00
Song B : 0.92
Song C : 0.84
Song D : 0.76
...
```

```text
Input Vector
      ↓
Cosine Similarity
      ↓
Similarity Scores
```

↓

## Step 6: Flatten Similarity Scores

Convert

```python
(50674,1)
```

into

```python
(50674,)
```

using

```python
similarity_scores = similarity_scores.ravel()
```

↓

## Step 7: Sort Similarities

```python
sorted_indices = np.argsort(similarity_scores)
```

Example:

```text
[8, 25, 10, 3, 1025]
```

Highest similarities are at the end.

↓

## Step 8: Retrieve Top-K Similar Songs

Reverse and select Top K:

```python
top_indices = sorted_indices[::-1][1:k+1]
```

Exclude:

- Self recommendation

Example:

```text
[523, 102, 876, 402, 98]
```

```text
Similarity Scores
        ↓
Sort
        ↓
Top K Indices
```

↓

## Step 9: Fetch Song Metadata

Retrieve the rows corresponding to Top K indices.

```python
songs_df.iloc[top_indices]
```

Output:

| Song Name | Artist |
|------------|--------|
| Hips Don't Lie | Shakira |
| La Tortura | Shakira |
| Whenever Wherever | Shakira |
| Chantaje | Shakira |
| She Wolf | Shakira |

↓

## Step 10: Return Recommendations

Final output:

```text
User Input

Waka Waka - Shakira

↓

Recommended Songs

1. Hips Don't Lie
2. Whenever Wherever
3. La Tortura
4. Chantaje
5. She Wolf
```

---

# Complete Pipeline

```text
Song Name + Artist Name
            ↓
Metadata Dataset
            ↓
Matching Row
            ↓
Index
            ↓
Transformed Matrix
            ↓
Input Vector
            ↓
Cosine Similarity
            ↓
Similarity Scores
            ↓
Sort Similarities
            ↓
Top K Indices
            ↓
Song Metadata Dataset
            ↓
Recommended Songs
```

---

# Feature Engineering Pipeline

```text
Songs Dataset
        ↓
Frequency Encoding
        ↓
One Hot Encoding
        ↓
TF-IDF Vectorization
        ↓
Scaling
        ↓
Transformed Matrix
        ↓
Content-Based Recommendation
```

---

# Answer

> When the user provides a song name and artist name, I first retrieve the corresponding row from the songs metadata dataset and obtain its index. Since all songs have already been transformed into feature vectors using TF-IDF, One Hot Encoding, Frequency Encoding, and scaling, I use the index to extract the corresponding input vector from the transformed matrix. I then compute cosine similarity between this vector and all other songs. After sorting the similarity scores, I select the Top-K most similar songs and retrieve their metadata from the original dataset to generate recommendations.